In [ ]:
import numpy as np
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mlflow
import mlflow.sklearn

In [28]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
867,"Wow, I can't believe i'm the first and only on...",positive
210,<br /><br />Robot jox is a great little film o...,positive
804,I first flicked onto the LoG accidentally one ...,positive
733,I can't believe that anyone would green light ...,negative
277,I wish I would have read more reviews and more...,negative


In [29]:
# define text preprocessing functions

def lemmatization(text: str) -> str:
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text: str) -> str:
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text: str) -> str:
    """Remove numbers from the text."""
    text = "".join([char for char in text if not char.isdigit()])
    return text

def lower_case(text: str) -> str:
    """Convert text to lowercase."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text: str) -> str:
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text: str) -> str:
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.|S+')
    return url_pattern.sub(r'', text)

def normalize_text(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise


In [30]:
df = normalize_text(df)
df.head()

,review,sentiment
867,wow can t believe first one post comment great...,positive
210,br br robot jox great little film ok set bad a...,positive
804,first flicked onto log accidentally one night ...,positive
733,can t believe anyone would green light let alo...,negative
277,wish would read review opinion movie rented it...,negative


In [31]:
df['sentiment'].value_counts()

sentiment
negative    267
positive    233
Name: count, dtype: int64

In [32]:
x = df['sentiment'].isin(['positive', 'negative'])
df = df[x]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 867 to 508
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     500 non-null    object
 1   sentiment  500 non-null    object
dtypes: object(2)
memory usage: 11.7+ KB


In [33]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

,review,sentiment
867,wow can t believe first one post comment great...,1
210,br br robot jox great little film ok set bad a...,1
804,first flicked onto log accidentally one night ...,1
733,can t believe anyone would green light let alo...,0
277,wish would read review opinion movie rented it...,0


In [34]:
df.isnull().sum()


review       0
sentiment    0
dtype: int64

In [35]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [37]:
import dagshub
mlflow.set_tracking_uri('https://dagshub.com/Sharif-Abusad/sentiment-analysis-mlops.mlflow')
dagshub.init(repo_owner='Sharif-Abusad', repo_name='sentiment-analysis-mlops', mlflow=True)

mlflow.set_experiment('Logistic Regression Baseline')

Accessing as Sharif-Abusad

Initialized MLflow to track repo "Sharif-Abusad/sentiment-analysis-mlops"

Repository Sharif-Abusad/sentiment-analysis-mlops initialized!

2026/09/05 16:53:32 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/b7cf8a65cc144683bad9667fb699c271', creation_time=1788607413011, experiment_id='0', last_update_time=1788607413011, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [39]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression

# Configure Logging
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test size", 0.20)
        
        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)   # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2026-09-05 17:09:30,068 - INFO - Starting MLflow run...
2026-09-05 17:09:31,691 - INFO - Logging preprocessing parameters...
2026-09-05 17:09:32,797 - INFO - Initializing Logistic Regression model...
2026-09-05 17:09:32,799 - INFO - Fitting the model...
2026-09-05 17:09:32,827 - INFO - Model training complete.
2026-09-05 17:09:32,829 - INFO - Logging model parameters...
2026-09-05 17:09:33,173 - INFO - Making predictions...
2026-09-05 17:09:33,175 - INFO - Calculating evaluation metrics...
2026-09-05 17:09:33,192 - INFO - Logging evaluation metrics...
2026-09-05 17:09:35,096 - INFO - Saving and logging the model...
2026/09/05 17:09:44 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2026-09-05 17:09:44,747 - INFO - Model training and logging completed in 13.06 seconds.
2026-09-05 17:09:44,749 - INFO - Accuracy: 0.69
2026-09-05 17:09:44,749 - INFO - Precision: 0.6739130434782609
2026-09-